# MedLoRA · Zero-shot 基线 (Kaggle)

右侧 Settings → Accelerator 选 **GPU T4 x2**，Internet 打开。

跑完后 `outputs/eval/` 里的 json 就是基线数字，下载下来放进报告。

In [ ]:
REPO_URL = "https://github.com/AugustLoo/MedLoRA.git"  # 推到 GitHub 后改成你的地址
MODEL = "Qwen/Qwen2.5-VL-3B-Instruct"

!git clone -q $REPO_URL /kaggle/working/MedLoRA
%cd /kaggle/working/MedLoRA
!pip install -q -r requirements.txt
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# 数据
!python data/download_slake.py
!python data/download_pubmedqa.py

In [ ]:
# 先跑 20 条确认能通, 再跑全量
!python eval/eval_slake.py --model $MODEL --tag smoke --limit 20

In [ ]:
# 全量基线: 三张表 (SLAKE 约 1000 条 / TextVQA 300 条 / PubMedQA 1000 条)
!MODEL=$MODEL bash train/eval_all.sh baseline

In [ ]:
import json, glob
for f in sorted(glob.glob('outputs/eval/*_baseline.json')):
    print(f); print(json.dumps(json.load(open(f)), indent=2, ensure_ascii=False)[:800]); print()

## 下一步: SFT

```
!pip install -q "llamafactory[torch,metrics] @ git+https://github.com/hiyouga/LLaMA-Factory.git"
!bash train/run_sft.sh
!MODEL=$MODEL bash train/eval_all.sh sft_r16 outputs/sft_slake_qlora_r16
```
训练完把 `outputs/sft_slake_qlora_r16` 里的 adapter 文件 (几十 MB) 存成 Kaggle Dataset，免得会话结束丢失。